# Domain 3: Product and Model Selection
**Exam**: Claude Certified Associate – Foundations (CCAO-F)
**Weighting**: 12% of exam
**Depth**: deep

## Domain Overview

This domain covers choosing the right Claude product surface (chat, Projects, research
mode, artifacts) and the right model tier (Haiku, Sonnet, Opus) for a given task, plus managing
context/memory limitations sensibly.

**Why it matters**: picking the wrong tool wastes time and money — e.g. using the most expensive
model for a trivial task, or a bare chat window for a task that needs a persistent Project.

**Key concepts to know**: product feature fit, model tier tradeoffs (cost/speed/quality), and context
window / memory management (when to restart vs. summarize vs. persist).

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Select appropriate Claude product features

Use plain chat for one-off questions, Projects for recurring work with shared knowledge and
instructions, research mode for multi-source investigation, and artifacts for reusable documents,
code, or diagrams that will be edited or reused.

In [ ]:
product_fit = {
    "one-off question about a policy": "chat",
    "recurring weekly report using the same 3 source docs": "Project",
    "deep multi-source investigation of a market trend": "research mode",
    "a reusable onboarding checklist to edit over time": "artifact",
}
for scenario, feature in product_fit.items():
    print(f"{scenario:55s} -> {feature}")

### Differentiate between Claude model types

Haiku is fastest/cheapest, best for high-volume simple tasks. Sonnet balances cost and
capability for most day-to-day work. Opus is the most capable, best reserved for complex reasoning,
ambiguous judgment calls, or high-stakes tasks.

In [ ]:
model_guidance = {
    "haiku": "high volume, simple, latency-sensitive (e.g. tagging 1000 tickets)",
    "sonnet": "everyday business tasks: drafting, analysis, summarization",
    "opus": "complex reasoning, ambiguous judgment, high-stakes one-off tasks",
}
for model, guidance in model_guidance.items():
    print(f"{model:8s}: {guidance}")

### Align model selection with task requirements

The right choice balances cost, speed, and quality against what the task actually needs.
Over-provisioning (Opus for a trivial task) wastes budget; under-provisioning (Haiku for a nuanced
judgment call) risks poor output.

In [ ]:
def recommend_model(volume: str, complexity: str) -> str:
    if volume == "high" and complexity == "low":
        return "haiku"
    if complexity == "high":
        return "opus"
    return "sonnet"

print(recommend_model("high", "low"))
print(recommend_model("low", "high"))
print(recommend_model("low", "medium"))

### Understand and manage context limitations and memory considerations

Long conversations eventually fill the context window, causing Claude to lose track of
earlier details. Options: summarize key facts periodically, persist important context in a Project's
knowledge/instructions, or start a fresh conversation with a concise recap.

In [ ]:
conversation_length_tokens = 45000
context_window_limit = 50000

def context_action(used: int, limit: int) -> str:
    ratio = used / limit
    if ratio > 0.85:
        return "summarize key facts now, or restart with a recap"
    if ratio > 0.6:
        return "keep going, but start tracking key facts for a summary soon"
    return "no action needed"

print(context_action(conversation_length_tokens, context_window_limit))

## Mini Hands-On Project

**Covers**: all Domain 3 sub-objectives.

**Scenario**: Your team runs three different kinds of Claude work: (1) tagging 2,000 support
tickets by category, (2) a weekly competitor-pricing research brief, (3) a long-running onboarding
Project with shared docs. Pick the right product feature and model for each, and simulate hitting a
context limit in the long-running one.

### Step 1: Match each workload to a product feature and model

In [ ]:
workloads = {
    "tag 2000 support tickets": {},
    "weekly competitor pricing brief": {},
    "long-running onboarding project": {},
}
# TODO-style fill-in for the mini-project narrative (already solved here as a worked example)
workloads["tag 2000 support tickets"] = {"feature": "chat/API", "model": "haiku"}
workloads["weekly competitor pricing brief"] = {"feature": "research mode", "model": "sonnet"}
workloads["long-running onboarding project"] = {"feature": "Project", "model": "sonnet"}
for w, choice in workloads.items():
    print(w, "->", choice)

### Step 2: Simulate approaching the context limit in the Project

In [ ]:
used_tokens = 42000
limit_tokens = 50000
print(context_action(used_tokens, limit_tokens))

key_facts_summary = call_claude("Summarize the 5 most important facts a new hire needs from this onboarding Project so far.")
print(key_facts_summary)

## Lab Exercises

### Exercise 1: Feature-fit matching

**Objective**: Match 6 new scenarios to the best Claude product feature.

**Produce**: A dict mapping each scenario to a feature with a one-line reason.

In [ ]:
new_scenarios = [
    "Quick unit conversion question",
    "Ongoing brand voice guide shared across 5 writers",
    "Comparing 4 vendors' public pricing pages",
    "One-time SQL query to debug an issue",
    "A living style guide document edited weekly",
    "Analyzing one uploaded spreadsheet once",
]
# TODO: map each scenario to chat / Project / research mode / artifact, with a one-line reason
mapping = {}
for s in new_scenarios:
    mapping[s] = ("...", "...")  # (feature, reason)
for s, (feature, reason) in mapping.items():
    print(s, "->", feature, "--", reason)

# Expected output: 6 sensible feature choices with justifications; recurring/shared-context items map to Project.

In [ ]:
# STRETCH GOAL: for the 2 scenarios you chose "artifact" for, generate the artifact content itself.
for s, (feature, reason) in mapping.items():
    if feature == "artifact":
        print(s, "->", call_claude(f"Draft the content for: {s}")[:150], "...")

# Expected output: Generated draft content for each artifact-tagged scenario.

### Exercise 2: Model tier recommender

**Objective**: Extend `recommend_model` to also factor in a 'stakes' parameter.

**Produce**: An updated function and test cases proving the new behavior.

In [ ]:
def recommend_model_v2(volume: str, complexity: str, stakes: str) -> str:
    # TODO: high stakes should push toward opus even at low complexity; keep prior volume/complexity logic otherwise
    pass

print(recommend_model_v2("high", "low", "low"))
print(recommend_model_v2("low", "low", "high"))

# Expected output: High-stakes low-complexity tasks recommend opus or sonnet, not haiku; other cases behave like v1.

In [ ]:
# STRETCH GOAL: add a cost estimate (relative units) per recommendation, e.g. haiku=1, sonnet=3, opus=10,
# and print total estimated relative cost for a batch of 5 mixed tasks.
tasks = [("high","low","low"), ("low","high","low"), ("low","low","high"), ("high","high","high"), ("low","low","low")]
cost_map = {"haiku": 1, "sonnet": 3, "opus": 10}
# TODO: sum cost_map[recommend_model_v2(*t)] for t in tasks
total_cost = None
print(total_cost)

# Expected output: A total relative cost number, with reasoning for which tasks drove it up.

### Exercise 3: Context budget tracker

**Objective**: Build a running tracker that recommends action as a conversation grows.

**Produce**: A function called at 3 points in a simulated conversation with 3 different recommendations.

In [ ]:
simulated_usage_points = [12000, 33000, 47000]
limit = 50000

# TODO: call context_action(used, limit) for each point and print the recommendation
for used in simulated_usage_points:
    pass

# Expected output: Three different recommendations reflecting increasing urgency as usage approaches the limit.

In [ ]:
# STRETCH GOAL: when the recommendation is to summarize, actually call Claude to produce that summary
# using a placeholder \"conversation so far\" string.
conversation_so_far = "Discussed onboarding steps 1-5, budget of $2000, deadline of March 1."
# TODO: if usage > 0.85 * limit, call call_claude to summarize conversation_so_far
summary = None
print(summary)

# Expected output: A generated summary only produced when usage crosses the 0.85 threshold.

### Exercise 4: Cost vs. quality tradeoff memo

**Objective**: Write a short internal memo justifying a model choice for a real workload.

**Produce**: A memo explaining the tradeoffs of using Sonnet instead of Opus for a specific task.

In [ ]:
workload_description = "Classify 5,000 inbound leads as hot/warm/cold based on a 2-sentence description each."

# TODO: build a prompt asking Claude to write a short internal memo justifying sonnet (not opus) for this workload,
# referencing cost, speed, and quality tradeoffs explicitly
memo_prompt = "..."
print(call_claude(memo_prompt))

# Expected output: A memo that explicitly references cost, speed, and quality tradeoffs relevant to the volume of the task.

In [ ]:
# STRETCH GOAL: also produce a counter-memo arguing FOR opus, then have Claude adjudicate which
# argument is stronger for this specific use case.
counter_memo = call_claude("Argue for using Opus instead for this same workload: " + workload_description)
adjudication = call_claude(f"Given these two memos, which is the stronger recommendation and why?\n1) {workload_description}\n2) {counter_memo}")
print(adjudication)

# Expected output: A reasoned adjudication that sides with sonnet given the stated volume, or clearly explains why not.

### Exercise 5: Long-conversation recap

**Objective**: Simulate restarting a stalled long conversation with a clean recap.

**Produce**: A concise recap prompt that could seed a brand-new conversation.

In [ ]:
stalled_conversation_summary_points = [
    "Client wants a 3-tier pricing page",
    "Budget confirmed at $5k for design",
    "Deadline is end of next month",
    "Two competitor examples were already reviewed",
]

# TODO: build a single recap prompt combining these points that could start a fresh conversation
recap_prompt = "..."
print(call_claude(recap_prompt))

# Expected output: A clean, self-contained recap that a new conversation could use as its starting context.

In [ ]:
# STRETCH GOAL: compress the recap to under 60 words without losing any of the 4 key points.
compressed = call_claude(f"Compress this to under 60 words, keeping all 4 facts: {recap_prompt}")
print(compressed, len(compressed.split()))

# Expected output: A version of the recap under 60 words, suitable for pasting into a chat window quickly.

## Self-Assessment: Product and Model Selection

Answer each question, then check the Answers cell below.


**Q1.** Which Claude product feature is best suited to a recurring task that reuses the same instructions and source documents every week?

**Q2.** For a high-volume, low-complexity task like tagging thousands of short tickets, which model tier is usually most appropriate?

**Q3.** What is the risk of using Opus for every task regardless of complexity?

**Q4.** What should you do when a long conversation is close to filling the context window and you still need it to continue accurately?

**Q5.** True or False: model selection should be based solely on which model produces the 'best' output, ignoring cost and speed.

## Answers


**A1.** A Project — it persists instructions and knowledge sources across sessions.

**A2.** Haiku — it is optimized for fast, low-cost, high-volume, simple tasks.

**A3.** Unnecessary cost and latency for tasks that don't need that level of capability.

**A4.** Summarize the key facts so far, or restart the conversation with a concise recap of what matters.

**A5.** False — model selection should balance cost, speed, and quality against the task's actual requirements.

---
**Next**: [Domain 4: Workflow Integration and Solution Design](./04-workflow-integration-and-solution-design.ipynb)
**Back**: [README](./README.md)